# Linear Regression — scikit-learn

Now that we understand the math, we use scikit-learn's `LinearRegression` which
wraps the Normal Equation (via LAPACK) and adds conveniences like feature pipelines.

**Topics:**
- `LinearRegression`, `Ridge`, `Lasso`
- Train/test split, cross-validation
- Feature scaling
- Regularisation to prevent overfitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

import sys
sys.path.insert(0, '../..')
from src.utils import set_style, regression_report

set_style()
np.random.seed(42)

## 1. Synthetic Multi-Feature Dataset

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=200,
    n_features=5,
    n_informative=3,   # only 3 of 5 features actually matter
    noise=20,
    random_state=42,
)

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'y range: [{y.min():.1f}, {y.max():.1f}]')

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 3. OLS Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print(f'Intercept : {lr.intercept_:.4f}')
print(f'Coefs     : {lr.coef_}')
print()
print('Test metrics:')
regression_report(y_test, y_pred)

## 4. Feature Scaling + Pipeline

Many algorithms are sensitive to feature scale (gradient descent converges
faster; regularisation penalties are fair). `StandardScaler` transforms:

$$x' = \frac{x - \mu}{\sigma}$$

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])

pipe.fit(X_train, y_train)
y_pred_pipe = pipe.predict(X_test)

print('Pipeline (scaled) test metrics:')
regression_report(y_test, y_pred_pipe)

## 5. Regularisation: Ridge (L2) and Lasso (L1)

Regularisation adds a penalty to the cost function to prevent overfitting:

| Method | Cost | Effect |
|--------|------|--------|
| Ridge | $J + \lambda\|\boldsymbol{\theta}\|_2^2$ | Shrinks all coefficients smoothly |
| Lasso | $J + \lambda\|\boldsymbol{\theta}\|_1$ | Drives some coefficients to exactly 0 (feature selection) |

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

ridge_r2 = []
lasso_r2 = []

for a in alphas:
    ridge = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=a))])
    lasso = Pipeline([('s', StandardScaler()), ('m', Lasso(alpha=a, max_iter=5000))])

    ridge.fit(X_train, y_train)
    lasso.fit(X_train, y_train)

    ridge_r2.append(r2_score(y_test, ridge.predict(X_test)))
    lasso_r2.append(r2_score(y_test, lasso.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(alphas, ridge_r2, 'o-', color='#2563EB', lw=2, label='Ridge (L2)')
ax.semilogx(alphas, lasso_r2, 's--', color='#DC2626', lw=2, label='Lasso (L1)')
ax.set_xlabel('Regularisation strength α')
ax.set_ylabel('Test R²')
ax.set_title('Ridge vs Lasso — Effect of Regularisation')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Cross-Validation

In [ ]:
cv_scores = cross_val_score(
    Pipeline([('s', StandardScaler()), ('m', LinearRegression())]),
    X, y,
    cv=5,
    scoring='r2',
)

print('5-fold CV R² scores:', np.round(cv_scores, 4))
print(f'Mean R²: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')

## 7. Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.6, color='#2563EB')
axes[0].axhline(0, color='#DC2626', lw=1.5)
axes[0].set_xlabel('Predicted ŷ')
axes[0].set_ylabel('Residual (y - ŷ)')
axes[0].set_title('Residuals vs Predicted')

# Distribution of residuals — should be ~N(0, σ²)
axes[1].hist(residuals, bins=20, color='#2563EB', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## Summary

| Tool | When |
|------|------|
| `LinearRegression` | Small dataset, no multicollinearity |
| `Ridge` | Correlated features, keep all |
| `Lasso` | Feature selection needed |
| `StandardScaler` in `Pipeline` | Always when using regularisation |
| `cross_val_score` | Unbiased performance estimate |

**Next:** [Apply to Kaggle World Happiness dataset →](04_kaggle_world_happiness.ipynb)